# Email Campaign A/B/C Test Analysis (Hillstrom)

## 1. Problem Statement

### Context
An online clothing retailer ran an email marketing experiment. 64,000 customers who had
purchased within the last 12 months were randomly split into three groups of about 21,300 each:
- **Mens E-Mail** — received an email featuring men's merchandise;
- **Womens E-Mail** — received an email featuring women's merchandise;
- **No E-Mail** — received no email (control group).

For two weeks after the campaign, the retailer tracked whether each customer visited the website,
made a purchase, and how much they spent.

### Business Question
Should the company send the email campaign, and if so, which version: men's or women's?

### Metrics
| Metric | Formula | Role |
|---|---|---|
| Purchase conversion rate | customers who purchased / all customers in the group | **primary** |
| Visit rate | customers who visited the website / all customers in the group | secondary |
| Revenue per customer | total spend in the group / all customers in the group | secondary |

**Why conversion is the primary metric:** the business earns money from purchases, not visits.
Revenue matters, but most customers spend $0, so the variance is high and a test on revenue
is less sensitive. The decision is based on conversion; revenue is analyzed as a supporting metric.

### Hypotheses
p_M, p_W, p_C — purchase conversion rate in the whole customer population with the men's email,
the women's email, and no email.

| Comparison | H₀ | H₁ |
|---|---|---|
| Men's vs Control | p_M = p_C | p_M ≠ p_C |
| Women's vs Control | p_W = p_C | p_W ≠ p_C |
| Men's vs Women's | p_M = p_W | p_M ≠ p_W |

### Significance Level
- α = 0.05, number of comparisons m = 3.
- Bonferroni correction: threshold for each comparison α / m = 0.05 / 3 ≈ **0.0167**.
  Without the correction, the chance of at least one false positive would be 1 − 0.95³ ≈ 14%.
- **Two-sided** tests: an email can either increase or decrease conversion,
  and when comparing the two emails we do not know in advance which one is better.

### Analysis Plan
1. Data quality check
2. Descriptive statistics
3. Randomization check (SRM, group balance)
4. Test power (MDE)
5. Proportion tests: conversion and visits
6. Revenue tests: t-test, bootstrap, Mann–Whitney
7. Segment analysis: which customers respond to which email
8. Business conclusion and recommendation

## 2. Data Quality Check

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

In [2]:
df = pd.read_csv('../data/hillstrom.csv')

In [3]:
df.shape

(64000, 12)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          64000 non-null  int64  
 1   history_segment  64000 non-null  str    
 2   history          64000 non-null  float64
 3   mens             64000 non-null  int64  
 4   womens           64000 non-null  int64  
 5   zip_code         64000 non-null  str    
 6   newbie           64000 non-null  int64  
 7   channel          64000 non-null  str    
 8   segment          64000 non-null  str    
 9   visit            64000 non-null  int64  
 10  conversion       64000 non-null  int64  
 11  spend            64000 non-null  float64
dtypes: float64(2), int64(6), str(4)
memory usage: 5.9 MB


In [5]:
df.isna().sum()

recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
segment            0
visit              0
conversion         0
spend              0
dtype: int64

In [6]:
df['recency'].between(1, 12).all() # should be 1-12 (months in year)

np.True_

In [7]:
df[(df['history'] < 0) | (df['spend'] < 0)]['recency'].count()

np.int64(0)

In [8]:
for col in ['mens', 'womens', 'newbie', 'visit', 'conversion']:
    print(col, df[col].isin([0, 1]).all())

mens True
womens True
newbie True
visit True
conversion True


In [9]:
for col in ['segment', 'zip_code', 'channel', 'history_segment']:
    print(col, list(df[col].unique()))

segment ['Womens E-Mail', 'No E-Mail', 'Mens E-Mail']
zip_code ['Surburban', 'Rural', 'Urban']
channel ['Phone', 'Web', 'Multichannel']
history_segment ['2) $100 - $200', '3) $200 - $350', '5) $500 - $750', '1) $0 - $100', '6) $750 - $1,000', '4) $350 - $500', '7) $1,000 +']


In [10]:
len(df[(df['spend'] > 0) & (df['conversion'] == 0)])

0

In [11]:
len(df[(df['spend'] == 0) & (df['conversion'] == 1)])

0

In [12]:
len(df[(df['visit'] == 0) & (df['conversion'] == 1)])

0

In [14]:
df.duplicated().sum()

np.int64(6562)

### Conclusion
- The dataset has **64,000 rows and 12 columns**; data types are correct and there are **no missing values**.
- All values are within valid ranges: `recency` is 1–12, `history` and `spend` are non-negative,
  binary columns contain only 0 and 1.
- Categorical columns have no unexpected values. Note: `zip_code` contains the typo `Surburban`
  (instead of *Suburban*) in the original data; it is kept as is.
- The data is logically consistent: `spend > 0` exactly when `conversion = 1`,
  and there are no purchases without a website visit.
- **6,562 rows are exact duplicates.** Since there is no customer ID and each row represents
  a separate customer, these are different customers with identical attributes, so they are kept.

**The data is clean and ready for analysis.**

## 3. Descriptive Statistics